# 03 · Congestión y ocupación
Ocupación horaria (+1 al entrar, −1 al salir) y patrón de llegadas.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, load, quality, flow, features, models

# Fuente de datos:
#   "db"     -> base de datos del reto (FortiClient VPN conectada + archivo .env)
#   "raw"    -> archivos exportados en data/raw
#   "sample" -> datos sintéticos de prueba (scripts/generar_datos_sinteticos.py)
SOURCE = "db"
DATA_DIR = ROOT / "data" / ("sample" if SOURCE == "sample" else "raw")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
if SOURCE == "db":
    from src import db
    db.test_connection()
    t = db.load_all()   # copia local en data/interim; db.load_all(refresh=True) para volver a descargar
else:
    t = load.load_all(DATA_DIR)

In [ ]:
vo = t['visit_occurrence']
vo['tipo_visita'] = vo['visit_concept_id'].map(config.VISIT_CONCEPTS).fillna(vo['visit_concept_id'].astype(str))
occ = flow.census(vo, 'visit_start_datetime', 'visit_end_datetime', freq='h')
occ_tipo = flow.census(vo, 'visit_start_datetime', 'visit_end_datetime', freq='h', group='tipo_visita')
arr = flow.arrivals(vo, 'visit_start_datetime', freq='h')

In [ ]:
ax = occ.set_index('fecha_hora')['ocupacion'].resample('D').max().plot(figsize=(12,3), label='Pico diario')
ax.axhline(occ['ocupacion'].quantile(.9), ls='--', c='r', label='P90 (umbral saturación)')
ax.legend(); ax.set_title('Ocupación máxima diaria');

## Heatmaps hora × día

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14,5))
sns.heatmap(flow.hour_weekday_matrix(vo, 'visit_start_datetime'), cmap='Blues', ax=axs[0]).set_title('Llegadas')
o = occ.assign(fecha_hora=occ['fecha_hora'])
sns.heatmap(flow.hour_weekday_matrix(o, 'fecha_hora', 'ocupacion', 'mean'), cmap='Reds', ax=axs[1]).set_title('Ocupación media')
plt.tight_layout()

## ¿La congestión alarga la estancia?

In [ ]:
vo['los_h'] = (vo['visit_end_datetime']-vo['visit_start_datetime']).dt.total_seconds()/3600
vo['ocupacion_llegada'] = flow.occupancy_at_arrival(vo, occ)
vo['q_ocup'] = pd.qcut(vo['ocupacion_llegada'], 5, labels=['Q1 baja','Q2','Q3','Q4','Q5 alta'], duplicates='drop')
vo.groupby(['tipo_visita','q_ocup'], observed=True)['los_h'].median().unstack().round(2)

## Guardar

In [ ]:
occ.to_csv(config.DATA_PROCESSED / 'ocupacion_horaria.csv', index=False)
occ_tipo.to_csv(config.DATA_PROCESSED / 'ocupacion_horaria_tipo.csv', index=False)
arr.to_csv(config.DATA_PROCESSED / 'llegadas_horarias.csv', index=False)